In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

/backup/workspace/github/agentic-ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00,   "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}


@tool
def get_product(name: str) -> str:
    """ Look up a product by name and return its price, rating, stock, and description. """
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {','.join(PRODUCTS)}"
    return str(p)


@tool
def get_review(name: str) -> str:
    """ Look up a product review by a product name and return the product name, number of reviews and rating """
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not found. Available: {','.join(REVIEWS)}"
    return str(r)

In [29]:
llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


agent = create_agent(
    llm_groq,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store."
)
agent2= create_agent(
    llm_groq,
    tools=[get_product,get_review],
    system_prompt="You are a helpful product assistant for an online tech store."
)

In [27]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)  

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [31]:
ask("What is the price of a wireless headphones?")
ask("What is the price of digital radio?")

The price of the wireless headphones is $79.99.
I couldn't find the price of a digital radio as it is not available in our store. We do have other products such as wireless headphones, smart watch, mechanical keyboard, and laptop stand. Would you like to know the price of any of these products?


In [ ]:
ask2("how do people like smart watch?")
ask2("how do people like mechanical keyboard?")
ask2("what is the price and review of smart watch?")

People seem to like smart watches, with an average rating of 3.9 out of 5 stars based on 340 reviews. The product description mentions that it tracks heart rate and sleep, has a 5-day battery life, and is water-resistant. The price of the smart watch is $199.99.
People seem to like mechanical keyboards, with an average rating of 4.8 out of 5 stars based on 67 reviews. The price of a mechanical keyboard can vary, but one example is $129 for a tenkeyless keyboard with Cherry MX Brown switches and per-key RGB lighting.
The price of the smart watch is $199.99. It has a rating of 3.9 out of 5 stars and 340 reviews. The description of the smart watch is that it tracks heart rate and sleep, has a 5-day battery, and is water-resistant.


In [35]:
# In memory 

from langgraph.checkpoint.memory import InMemorySaver
agent3=create_agent(
    llm_groq,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

In [46]:
def ask3(question: str):
    config={"configurable":{"thread_id":"user-alice-session-1"}}
    result=agent3.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config
    )
    print(result["messages"][-1].content)

In [50]:
ask3("where is the capital city of nepal")
ask3("what is the prices of smart watch?")

The capital city of Nepal is Kathmandu.
The price of the smart watch is $199.99. It has a rating of 3.9, 340 reviews, and its description is "Tracks heart rate and sleep. 5-day battery, water-resistant."


In [51]:
ask3("what about india?")
ask3("what about reviews of this product?")

The capital city of India is New Delhi.
The smart watch has 340 reviews with a rating of 3.9.
